In [ ]:
# Dataset setup
# Download/extract the emergency-vehicle dataset separately, then set DATASET_DIR below.
# Do not commit Kaggle API credentials, private datasets, videos, or model weights.
from pathlib import Path
DATASET_DIR = Path('../data/emergency-vehicles')  # expected to contain train/ and valid/ (or val/) folders


In [ ]:
# Optional: extract a dataset archive that you have permission to use.
from zipfile import ZipFile
ARCHIVE_PATH = Path('../data/emergency-vehicles.zip')
if ARCHIVE_PATH.exists():
    with ZipFile(ARCHIVE_PATH, 'r') as archive:
        archive.extractall(DATASET_DIR)
else:
    print('No local archive found; ensure DATASET_DIR already contains the dataset.')


In [ ]:
# Create the YOLO dataset configuration.
# Adjust these paths to match the directory structure of your downloaded dataset.
train_images_path = (DATASET_DIR / 'train' / 'images').resolve()
valid_images_path = (DATASET_DIR / 'valid' / 'images').resolve()
if not valid_images_path.exists():
    valid_images_path = (DATASET_DIR / 'val' / 'images').resolve()
data_yaml_path = (DATASET_DIR / 'data.yaml').resolve()
data_yaml_path.parent.mkdir(parents=True, exist_ok=True)
data_yaml_path.write_text(
    f"train: {train_images_path}\nval: {valid_images_path}\nnc: 2\nnames: ['ambulance', 'fire_truck']\n",
    encoding='utf-8'
)
print(data_yaml_path)


In [ ]:
%pip install -r ../requirements.txt


In [ ]:
from ultralytics import YOLO

# Fine-tune a pretrained YOLOv8 small model.
model = YOLO('yolov8s.pt')
model.train(data=str(data_yaml_path), epochs=50, imgsz=640, device='cpu')


In [ ]:
from ultralytics import YOLO
import cv2

# Set these paths to your locally stored trained weights and test video.
WEIGHTS_PATH = Path('runs/detect/train/weights/best.pt')
VIDEO_PATH = Path('../data/sample-video.mp4')
model = YOLO(str(WEIGHTS_PATH))
cap = cv2.VideoCapture(str(VIDEO_PATH))

counts = {'ambulance': 0, 'fire_truck': 0}
line_y = 300
confidence_threshold = 0.9

def crossed_line(center_y):
    return line_y - 10 < center_y < line_y + 10

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    for result in model(frame, stream=True):
        for box in result.boxes:
            cls_id = int(box.cls[0])
            confidence = float(box.conf[0])
            if confidence < confidence_threshold:
                continue
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            label = model.names[cls_id]
            center_y = (y1 + y2) // 2
            if crossed_line(center_y) and label in counts:
                counts[label] += 1
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, f'{label} {confidence:.2f}', (x1, max(y1 - 5, 15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    cv2.line(frame, (0, line_y), (frame.shape[1], line_y), (0, 0, 255), 2)
    y_offset = 30
    for vehicle, count in counts.items():
        cv2.putText(frame, f'{vehicle}: {count}', (10, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        y_offset += 30
    cv2.imshow('Emergency Vehicle Detection', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()
print(counts)
